## 6. Logging Basics

**`print()`** is fine for quick debugging, but in production code you should use the **`logging`** module. It gives you:

- **Log levels**-— filter messages by severity.
- **Timestamps & context** - who, when, and where.
- **Multiple destinations** - console, file, network.
- **On/off switch** - disable debug logs in production without removing code.

### Log levels (lowest → highest severity)
| Level | Value | Use when |
|-------|-------|----------|
| `DEBUG` | 10 | Detailed diagnostic info for developers |
| `INFO` | 20 | Confirmation that things are working |
| `WARNING` | 30 | Something unexpected, but program still works |
| `ERROR` | 40 | A serious problem — function could not complete |
| `CRITICAL` | 50 | Program may be unable to continue |

> The default level is `WARNING` - only WARNING, ERROR, and CRITICAL messages are shown unless you configure it otherwise.

In [1]:
import logging
import sys

# ---- Basic configuration ----
# force=True resets any previous config (needed in notebooks)
logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s [%(levelname)-8s] %(message)s",
    datefmt="%H:%M:%S",
    stream=sys.stdout,
    force=True
)

# Emit one message at each level
logging.debug("Detailed diagnostic — only useful while debugging")
logging.info("Application started successfully")
logging.warning("Config file missing — using defaults")
logging.error("Failed to connect to database")
logging.critical("Disk is full — cannot write data")

20:59:57 [DEBUG   ] Detailed diagnostic — only useful while debugging
20:59:57 [INFO    ] Application started successfully
20:59:57 [WARNING ] Config file missing — using defaults
20:59:57 [ERROR   ] Failed to connect to database
20:59:57 [CRITICAL] Disk is full — cannot write data


In [2]:
# ---- Named logger — best practice ----
logger = logging.getLogger("myapp.bank")

class LoggedAccount:
    def __init__(self, owner, balance=0):
        self.owner   = owner
        self.balance = balance
        logger.info("Account created for %s with balance ₹%s", owner, balance)

    def deposit(self, amount):
        if amount <= 0:
            logger.warning("Invalid deposit amount: %s", amount)
            raise ValueError("Deposit must be positive")
        self.balance += amount
        logger.info("Deposited ₹%s — new balance: ₹%s", amount, self.balance)

    def withdraw(self, amount):
        if amount > self.balance:
            logger.error(
                "Withdrawal of ₹%s failed — insufficient funds (₹%s)",
                amount, self.balance
            )
            raise ValueError("Insufficient funds")
        self.balance -= amount
        logger.info("Withdrew ₹%s — new balance: ₹%s", amount, self.balance)

acc = LoggedAccount("Purvi", 10000)
acc.deposit(5000)
try:
    acc.withdraw(20000)
except ValueError:
    pass
acc.withdraw(3000)

21:00:07 [INFO    ] Account created for Purvi with balance ₹10000
21:00:07 [INFO    ] Deposited ₹5000 — new balance: ₹15000
21:00:07 [ERROR   ] Withdrawal of ₹20000 failed — insufficient funds (₹15000)
21:00:07 [INFO    ] Withdrew ₹3000 — new balance: ₹12000


In [3]:
# ---- Logging exceptions with exc_info ----
def risky_parse(text):
    try:
        return int(text)
    except ValueError:
        # exc_info=True attaches the full traceback to the log record
        logger.error("Failed to parse %r as int", text, exc_info=True)
        return None

risky_parse("not_a_number")

21:00:13 [ERROR   ] Failed to parse 'not_a_number' as int
Traceback (most recent call last):
  File "C:\Users\Purvi jain\AppData\Local\Temp\ipykernel_31772\627753399.py", line 4, in risky_parse
    return int(text)
ValueError: invalid literal for int() with base 10: 'not_a_number'


In [4]:
# ---- Log to a file (log to both console AND file) ----
file_logger = logging.getLogger("file_demo")
file_logger.setLevel(logging.DEBUG)

# File handler
fh = logging.FileHandler("/tmp/app.log", mode="w")
fh.setLevel(logging.WARNING)   # only WARNING+ goes to file
fh.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
file_logger.addHandler(fh)

file_logger.debug("This goes to console only")
file_logger.warning("This goes to console AND file")
file_logger.error("This too goes to console AND file")

# Read back the file
with open("/tmp/app.log") as f:
    print("\n--- Contents of app.log ---")
    print(f.read())

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\tmp\\app.log'